# HW3 Applied: Zero-Shot Super Resolution (ZSSR)

**Course:** Modern Computer Vision (Technion, Spring 2026)

**Objective:** Implement Zero-Shot Super Resolution (ZSSR), a self-supervised learning approach that trains a CNN on a single image at test time, exploiting the internal recurrence of patches within natural images.

**Key Reference:**

> Shocher et al., ["Zero-Shot Super-Resolution using Deep Internal Learning"](https://openaccess.thecvf.com/content_cvpr_2018/papers/Shocher_Zero-Shot_Super-Resolution_Using_CVPR_2018_paper.pdf) (CVPR 2018)
>
> Project page: [http://www.wisdom.weizmann.ac.il/~vision/zssr/](http://www.wisdom.weizmann.ac.il/~vision/zssr/)


**Student ID** — fill in the code cell below:

In [ ]:
STUDENT_ID = ""

In [ ]:
# Setup and imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import math
from typing import Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Utility Functions


In [ ]:
def psnr(img1, img2):
    """
    Calculate PSNR (Peak Signal-to-Noise Ratio) between two images.
    Images should be in range [0, 1] or [0, 255].
    """
    if isinstance(img1, torch.Tensor):
        img1 = img1.detach().cpu().numpy()
    if isinstance(img2, torch.Tensor):
        img2 = img2.detach().cpu().numpy()
    
    if img1.max() > 2.0:   
        img1 = img1 / 255.0
    if img2.max() > 2.0:
        img2 = img2 / 255.0
    img1 = np.clip(img1, 0, 1)  
    img2 = np.clip(img2, 0, 1)
    
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(1.0 / np.sqrt(mse))


def resize_bicubic(image_tensor, scale_factor=None, size=None):
    """
    Resize an image using bicubic interpolation.
    
    Args:
        image_tensor: torch tensor of shape (C, H, W) or (1, C, H, W)
        scale_factor: float, scale multiplier
        size: tuple (H, W) for target size
    
    Returns:
        Resized tensor
    """
    ndim = image_tensor.dim()
    if ndim == 2:                       # (H, W) grayscale
        image_tensor = image_tensor.unsqueeze(0).unsqueeze(0)
    elif ndim == 3:                     # (C, H, W)
        image_tensor = image_tensor.unsqueeze(0)
    
    in_h, in_w = image_tensor.shape[-2], image_tensor.shape[-1]
    if size is not None:
        downscale = (size[0] < in_h) or (size[1] < in_w)
    else:
        downscale = (scale_factor is not None) and (scale_factor < 1)

    resized = F.interpolate(
        image_tensor,
        scale_factor=scale_factor,
        size=size,
        mode='bicubic',
        align_corners=False,
        antialias=bool(downscale)
    ).clamp(0, 1)                       
    if ndim == 2:
        return resized.squeeze(0).squeeze(0)
    elif ndim == 3:
        return resized.squeeze(0)
    return resized


def load_test_images():
    """
    Load two grayscale test images for super-resolution experiments.
    Uses scikit-image standard images (camera, coins), with a synthetic fallback.
    """
    images = {}
    
    try:
        from skimage import data as skdata
        cam = skdata.camera().astype(np.float32) / 255.0
        images['Cameraman'] = torch.from_numpy(cam)
        
        # "Coins" — 303x384, good fine detail
        coins = skdata.coins().astype(np.float32) / 255.0
        # Crop to even dimensions for clean downsampling
        h, w = coins.shape
        h, w = h - h % 4, w - w % 4
        images['Coins'] = torch.from_numpy(coins[:h, :w])
        
        print("Loaded scikit-image test images (Cameraman, Coins)")
    except ImportError:
        print("scikit-image not found, generating synthetic test images")
        for name, seed in [('Image 1', 42), ('Image 2', 123)]:
            np.random.seed(seed)
            h, w = 256, 256
            img = np.zeros((h, w), dtype=np.float32)
            sq = 16
            checker = np.indices((h, w)).sum(axis=0) // sq
            img += 0.3 * ((checker % 2).astype(np.float32))
            gx = np.linspace(0, 1, w)
            gy = np.linspace(0, 1, h)
            img += 0.4 * (gx[None, :] + gy[:, None]) / 2
            y, x = np.ogrid[:h, :w]
            for _ in range(3):
                cy = np.random.randint(50, h-50)
                cx = np.random.randint(50, w-50)
                r = np.random.randint(20, 60)
                circle = ((x - cx)**2 + (y - cy)**2) <= r**2
                img[circle] = np.clip(img[circle] + 0.3, 0, 1)
            img += 0.05 * np.random.randn(h, w).astype(np.float32)
            img = np.clip(img, 0, 1)
            images[name] = torch.from_numpy(img)
    
    return images


def visualize_results(results, figsize=(16, 4)):
    """
    Visualize super-resolution results: HR, LR, Bicubic, ZSSR side by side.
    
    Args:
        results: dict with keys 'HR', 'LR', 'Bicubic', 'ZSSR',
                 'PSNR_Bicubic', 'PSNR_ZSSR'
    """
    titles = [
        ('HR', 'Ground Truth'),
        ('LR', 'Low-Res Input'),
        ('Bicubic', f"Bicubic  PSNR: {results['PSNR_Bicubic']:.2f} dB"),
        ('ZSSR', f"ZSSR  PSNR: {results['PSNR_ZSSR']:.2f} dB"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    for ax, (key, title) in zip(axes, titles):
        img = results[key]
        if isinstance(img, torch.Tensor):
            img = img.detach().cpu().numpy()
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    return fig

In [ ]:
def plot_training_curves(loss_history, psnr_history, figsize=(14, 5)):
    """
    Plot training loss and PSNR curves.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Loss curve
    ax1.plot(loss_history, linewidth=2, label='Training Loss')
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Loss (L1)', fontsize=11)
    ax1.set_title('Training Loss Over Time', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # PSNR curve
    ax2.plot(psnr_history, linewidth=2, color='green', label='Training PSNR')
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('PSNR (dB)', fontsize=11)
    ax2.set_title('PSNR Over Time', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    return fig


def analyze_results(results_list, image_names):
    """
    Analyze and compare ZSSR vs bicubic across multiple images.
    
    Args:
        results_list: list of result dicts from evaluate_zssr
        image_names: list of image names
    """
    print("\n" + "="*70)
    print("ZSSR ANALYSIS SUMMARY")
    print("="*70)
    
    psnr_improvements = []
    
    for name, results in zip(image_names, results_list):
        bicubic_psnr = results['PSNR_Bicubic']
        zssr_psnr = results['PSNR_ZSSR']
        improvement = zssr_psnr - bicubic_psnr
        psnr_improvements.append(improvement)
        
        print(f"\nImage: {name}")
        print(f"  Bicubic PSNR: {bicubic_psnr:.3f} dB")
        print(f"  ZSSR PSNR:    {zssr_psnr:.3f} dB")
        print(f"  Improvement:  {improvement:+.3f} dB")
    
    avg_improvement = np.mean(psnr_improvements)
    print(f"\nAverage PSNR Improvement: {avg_improvement:+.3f} dB")
    print("="*70)
    
    return psnr_improvements


# Part 1: Understanding ZSSR

## The ZSSR Concept

### Training Phase
Given a low-resolution image **I_LR**:
1. **Downscale** I_LR by a factor (e.g., 2) to get **I_LR_downscaled**
2. Train a CNN to map: **I_LR_downscaled** → **I_LR**
3. The model learns to "upscale" this degraded version back to the original

### Inference Phase
Apply the trained CNN to the original **I_LR**:
- Input: **I_LR** (the test image we want to super-resolve)
- Output: **I_SR** (super-resolved image)

### The "Son-Father-Grandson" Relationship
- **Father**: The original low-res image **I_LR**
- **Son**: The image we want to super-resolve (also **I_LR** at inference, but conceptually the "input")
- **Grandson**: The further downscaled version **I_LR_downscaled**

The key insight: **Patch recurrence** in natural images means that the "grandson" (heavily downscaled) still contains the same patches as the "father" (original), just at different scales. The CNN learns to exploit this self-similarity.

### Why It Works
Internal patch recurrence means small patches appear repeatedly within a single image. When we train on a *single* image, we're not learning general upsampling rules—we're learning the specific internal structures *of that particular image*. This is surprisingly effective for super-resolution!


## Question: Patch Recurrence

**In your own words:** Explain why the internal recurrence of patches within a single natural image enables zero-shot super resolution without requiring a large dataset.

*Your answer here:*

# Part 2: Basic ZSSR Model

Implement a simple CNN-based architecture that:
1. Takes a low-res image and a scale factor as input
2. Bicubically upsamples to the target high-res size
3. Refines the upsampled image using convolutional layers
4. Uses a residual connection (learn the residual, not the full image)


In [ ]:
class ZSSRNet(nn.Module):
    """
    Zero-Shot Super Resolution Network.
    
    Architecture:
    - Input: upscaled image via bicubic interpolation
    - Residual CNN: learns the residual (difference from bicubic)
    - Output: bicubic_upscaled + residual
    
    Args:
        n_channels: number of feature channels (default: 64)
        n_layers: number of convolutional layers (default: 8)
        kernel_size: size of conv kernels (default: 3)
    """
    
    def __init__(self, n_channels=64, n_layers=8, kernel_size=3):
        raise NotImplementedError("Students should implement this")
    
    def forward(self, x, scale_factor=2, target_size=None):
        """
        Forward pass for ZSSR.
        
        Args:
            x: input tensor of shape (B, 1, H, W) or (1, H, W)
            scale_factor: upsampling factor (e.g., 2 for 2x)
            target_size: optional (H, W) tuple for target size
        
        Returns:
            super-resolved output tensor
        """
        raise NotImplementedError("Students should implement this")

# Part 3: Dataset

Implement a dataset class that:
1. Takes a single image
2. Generates pairs of (HR, LR) where LR is a downscaled version of HR
3. Applies random crops for training diversity
4. Optionally applies **8-fold geometric augmentation** when `augment=True`:
   - the 8 variants are **4 rotations** (0, 90, 180, 270 degrees) x **2 flips** (no flip, horizontal flip)
   - apply **one** randomly-chosen variant to each crop (not all 8 at once)

In [ ]:
class ZSSRDataset(Dataset):
    """
    Dataset for Zero-Shot Super Resolution on a single image.
    
    The "father–grandson" idea:
      - Crop a patch of size crop_size from the original image  →  this is the HR target
      - Downscale it by scale_factor using bicubic interpolation  →  this is the LR input
    
    __getitem__ should return a dict:
        {'HR': tensor (crop_size, crop_size),
         'LR': tensor (crop_size // scale_factor, crop_size // scale_factor)}
    
    Note: HR and LR have *different* spatial dimensions!
    
    Args:
        image: single image as torch tensor of shape (H, W)
        scale_factor: downsampling factor (e.g., 2 for 2x downsampling)
        crop_size: size of random HR crops to extract (default: 64)
        n_samples: number of random crops per epoch (default: 100)
        augment: if True, apply 8-fold geometric augmentation — randomly transform each crop
                 with ONE of: {0,90,180,270}-degree rotation x {no-flip, horizontal-flip}
                 (8 possible variants; one per sample, not all 8). This is training-time
                 augmentation that makes the model robust to orientation. (default: False)
    """
    
    def __init__(self, image, scale_factor=2, crop_size=64, n_samples=100, augment=False):
        raise NotImplementedError("Students should implement this")
    
    def __len__(self):
        raise NotImplementedError("Students should implement this")
    
    def __getitem__(self, idx):
        raise NotImplementedError("Students should implement this")

# Part 4: Training Loop

Implement the training function with:
1. L1 loss (or MSE)
2. Adam optimizer with learning rate scheduling
3. PSNR logging during training
4. Progress tracking


In [ ]:
def train_zssr(
    model,
    dataloader,
    scale_factor=2,
    num_epochs=100,
    learning_rate=0.001,
    device='cpu',
    verbose=True
):
    """
    Train the ZSSR model on a single image.
    
    The dataloader yields batches of {'HR': (B, crop, crop), 'LR': (B, crop//sf, crop//sf)}.
    Note that LR and HR have different spatial sizes — the model upsamples LR to match HR.
    
    Hints:
    - Use L1 loss (nn.L1Loss) or MSE
    - Use Adam optimizer
    - Remember to unsqueeze a channel dim before passing to the model:
      lr.unsqueeze(1) → (B, 1, H_lr, W_lr)
    - Pass target_size=(H_hr, W_hr) to the model so it knows the output resolution
    
    Args:
        model: ZSSRNet instance
        dataloader: DataLoader for the single image
        scale_factor: upsampling factor
        num_epochs: number of training epochs
        learning_rate: initial learning rate
        device: 'cpu' or 'cuda'
        verbose: whether to print progress
    
    Returns:
        model: trained model
        loss_history: list of training losses
        psnr_history: list of training PSNR values
    """
    raise NotImplementedError("Students should implement this")

# Part 5: Evaluation

Evaluate the model by:
1. Computing PSNR against ground truth
2. Comparing with bicubic baseline
3. Visualizing results


In [ ]:
def evaluate_zssr(
    model,
    image_hr,
    scale_factor=2,
    device='cpu',
    title="ZSSR Evaluation"
):
    """
    Evaluate ZSSR model on a single image.
    
    Args:
        model: trained ZSSRNet
        image_hr: ground truth high-res image (H, W)
        scale_factor: downsampling factor
        device: 'cpu' or 'cuda'
        title: title for visualization
    
    Returns:
        results: dict with PSNR values and images
    """
    raise NotImplementedError("Students should implement this")

# Part 6: Self-Ensemble — Advanced

In Part 3 your `ZSSRDataset` already applies **8-fold geometric augmentation** when `augment=True` — the 8 variants are **4 rotations** (0, 90, 180, 270 degrees) x **2 flips** (no flip, horizontal flip). That augmentation acts at **training time**: it makes the *model* robust to orientation.

Now implement the **test-time** counterpart — the **self-ensemble** (`ensemble_predict_zssr`, below):
- take the input image, build all 8 transformed versions,
- super-resolve each, **invert the transform** (un-rotate / un-flip),
- and average the 8 results into one prediction.

**Why self-ensemble at all?** A conv network is not perfectly rotation/flip-equivariant, so it produces *slightly different errors* on each of the 8 views of the same image. Once you un-transform them, you have 8 estimates of the **same** HR image whose errors are partly independent — so averaging them cancels much of that error: the real detail (shared by all 8) reinforces, while the per-view mistakes average out. It's the same reason an average of many noisy measurements beats a single one — a small, free accuracy/cleanliness boost at test time, with no extra training (and a bigger one when each view's errors are larger, e.g. harder degradations).

**Why are augmentation and the self-ensemble complementary?** Averaging only helps if all 8 views are *individually good*. That is exactly what training-time augmentation buys: a model trained with `augment=True` is good at **every** orientation, so all 8 predictions are reliable and their independent errors cancel. Without augmentation the model is good only at the original orientation — the 7 rotated/flipped views are out-of-distribution and wrong, so averaging them in actually **hurts** (a broken or un-augmented ensemble scores *below* a single pass). So augmentation (training) makes the 8 views trustworthy; the self-ensemble (test) cashes them in. You stack them — you don't choose between them.

In [ ]:
def ensemble_predict_zssr(model, image_lr, scale_factor=2, device='cpu'):
    """
    Predict using geometric self-ensemble: average predictions from 8 augmented versions.
    
    Args:
        model: trained ZSSRNet
        image_lr: low-res image (H, W)
        scale_factor: upsampling factor
        device: 'cpu' or 'cuda'
    
    Returns:
        ensemble_sr: averaged super-resolved image
    """
    raise NotImplementedError("Students should implement this (bonus)")

### (Optional) Self-check for `ensemble_predict_zssr`

A quick sanity test for your self-ensemble. It trains a small throwaway model (with augmentation) on a crop (~30–60 s), then checks that your `ensemble_predict_zssr` returns the right (HR) size and is at least as good as plain bicubic, and shows a **Bicubic vs ZSSR vs self-ensemble vs ground-truth** comparison.

*Note: this trains with `augment=True`. That's deliberate — the self-ensemble is a **test-time** technique, and it only helps if the model was trained to be orientation-robust (**training-time** augmentation).*

In [ ]:
def selfcheck_ensemble(scale_factor=2, device='cpu'):
    from torch.utils.data import DataLoader
    img = next(iter(load_test_images().values())).float()
    c = min(160, img.shape[-2], img.shape[-1]); c -= c % scale_factor
    top = (img.shape[-2] - c) // 2; left = (img.shape[-1] - c) // 2
    hr = img[top:top + c, left:left + c]

    ds = ZSSRDataset(hr, scale_factor=scale_factor, crop_size=64, n_samples=100, augment=True)
    dl = DataLoader(ds, batch_size=4, shuffle=True)
    model = ZSSRNet(n_channels=64, n_layers=8)
    model, _, _ = train_zssr(model, dl, scale_factor=scale_factor,
                             num_epochs=60, device=device, verbose=False)
    model = model.to(device).eval()

    lr = resize_bicubic(hr.unsqueeze(0), scale_factor=1.0 / scale_factor).squeeze(0)
    bicubic = resize_bicubic(lr.unsqueeze(0), scale_factor=float(scale_factor)).squeeze(0)
    with torch.no_grad():
        single = model(lr.unsqueeze(0).unsqueeze(0).to(device),
                       scale_factor=scale_factor, target_size=(c, c)).squeeze().cpu().clamp(0, 1)
    sr = ensemble_predict_zssr(model, lr, scale_factor=scale_factor, device=device)

    print(f"[shape] ensemble output {tuple(sr.shape[-2:])} | expected {(c, c)} -> "
          f"{'OK' if tuple(sr.shape[-2:]) == (c, c) else 'WRONG SIZE'}")
    p_bic, p_single, p_ens = psnr(bicubic, hr), psnr(single, hr), psnr(sr, hr)
    print(f"[psnr]  bicubic {p_bic:.2f} dB | ZSSR {p_single:.2f} dB | self-ensemble {p_ens:.2f} dB "
          f"| delta vs bicubic {p_ens - p_bic:+.2f} dB")
    if p_ens >= p_bic - 0.3:
        print("        OK — ensemble output is consistent with the baseline.")
    else:
        print("        PROBLEM — your ensemble scores far below plain bicubic, so the 8")
        print("        predictions aren't being combined correctly. Re-read how the")
        print("        geometric self-ensemble is meant to work (see the ZSSR paper).")

    panels = [("Bicubic x2", bicubic, p_bic), ("ZSSR (single pass)", single, p_single),
              ("ZSSR (self-ensemble)", sr, p_ens), ("Ground truth (HR)", hr, None)]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    for ax, (title, im, p) in zip(axes, panels):
        ax.imshow(im.detach().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(title if p is None else f"{title}\nPSNR {p:.2f} dB")
        ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
selfcheck_ensemble(scale_factor=2, device=device)

# Part 7: Main Experiment

Let's run the complete ZSSR pipeline on test images.

In [ ]:
# Load test images
print("Loading test images...")
test_images = load_test_images()
image_names = list(test_images.keys())

print(f"Loaded {len(test_images)} images:")
for name, img in test_images.items():
    print(f"  {name}: {img.shape}")

# Visualize
fig, axes = plt.subplots(1, len(test_images), figsize=(5 * len(test_images), 4))
if len(test_images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, test_images.items()):
    ax.imshow(img.numpy(), cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Training on Image 1

In [ ]:
# Prepare data for first image
scale_factor = 2
img_name_1 = image_names[0]
test_image_1 = test_images[img_name_1]

dataset_1 = ZSSRDataset(
    test_image_1,
    scale_factor=scale_factor,
    crop_size=64,
    n_samples=200,
    augment=False
)

dataloader_1 = DataLoader(dataset_1, batch_size=4, shuffle=True)

# Initialize model
model_1 = ZSSRNet(n_channels=64, n_layers=8)
print(f"Model parameters: {sum(p.numel() for p in model_1.parameters()):,}")

# Train
print(f"\nTraining ZSSR on {img_name_1}...")
model_1, loss_hist_1, psnr_hist_1 = train_zssr(
    model_1,
    dataloader_1,
    scale_factor=scale_factor,
    num_epochs=100,
    learning_rate=0.001,
    device=device,
    verbose=True
)

print("\nTraining complete!")

In [ ]:
# Plot training curves
plot_training_curves(loss_hist_1, psnr_hist_1)
plt.show()


In [ ]:
# Evaluate on Image 1
print(f"Evaluating on {img_name_1}...")
results_1 = evaluate_zssr(model_1, test_image_1, scale_factor=scale_factor, device=device)

print(f"\nResults for {img_name_1}:")
print(f"  Bicubic PSNR: {results_1['PSNR_Bicubic']:.3f} dB")
print(f"  ZSSR PSNR:    {results_1['PSNR_ZSSR']:.3f} dB")
print(f"  Improvement:  {results_1['PSNR_ZSSR'] - results_1['PSNR_Bicubic']:+.3f} dB")

# Visualize
visualize_results(results_1)
plt.show()

## Training on Image 2

In [ ]:
# Prepare data for Image 2
img_name_2 = image_names[1]
test_image_2 = test_images[img_name_2]

dataset_2 = ZSSRDataset(
    test_image_2,
    scale_factor=scale_factor,
    crop_size=64,
    n_samples=200,
    augment=False
)

dataloader_2 = DataLoader(dataset_2, batch_size=4, shuffle=True)

# Initialize and train
model_2 = ZSSRNet(n_channels=64, n_layers=8)

print(f"Training ZSSR on {img_name_2}...")
model_2, loss_hist_2, psnr_hist_2 = train_zssr(
    model_2,
    dataloader_2,
    scale_factor=scale_factor,
    num_epochs=100,
    learning_rate=0.001,
    device=device,
    verbose=True
)

print("Training complete!")

In [ ]:
# Evaluate on Image 2
print(f"Evaluating on {img_name_2}...")
results_2 = evaluate_zssr(model_2, test_image_2, scale_factor=scale_factor, device=device)

print(f"\nResults for {img_name_2}:")
print(f"  Bicubic PSNR: {results_2['PSNR_Bicubic']:.3f} dB")
print(f"  ZSSR PSNR:    {results_2['PSNR_ZSSR']:.3f} dB")
print(f"  Improvement:  {results_2['PSNR_ZSSR'] - results_2['PSNR_Bicubic']:+.3f} dB")

# Visualize
visualize_results(results_2)
plt.show()

# Part 8: Analysis & Discussion

## Summary of Results


In [ ]:
# Analyze all results
results_list = [results_1, results_2]

psnr_improvements = analyze_results(results_list, image_names)

## Visualization Comparison


In [ ]:
# Create side-by-side comparison
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (results, name) in enumerate(zip([results_1, results_2], image_names)):
    hr = results['HR']
    lr = results['LR']
    bicubic = results['Bicubic']
    zssr = results['ZSSR']
    
    for img_data, ax, title in [
        (hr, axes[row, 0], f'{name}: HR Ground Truth'),
        (lr, axes[row, 1], f'{name}: LR Input'),
        (bicubic, axes[row, 2], f"Bicubic\nPSNR: {results['PSNR_Bicubic']:.2f} dB"),
        (zssr, axes[row, 3], f"ZSSR\nPSNR: {results['PSNR_ZSSR']:.2f} dB"),
    ]:
        if isinstance(img_data, torch.Tensor):
            img_data = img_data.detach().cpu().numpy()
        ax.imshow(img_data, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.show()

### Your analysis & discussion (write-up)

Using **your own** results from Parts 8–9, write a short analysis (a few sentences per point):

1. **Which image did ZSSR help most**, and by how many dB over bicubic? Relate this to patch recurrence / the image's texture and structure.
2. **Where did ZSSR struggle** (smallest gain over bicubic)? What property of that image explains it?
3. **What do your training curves show** about how the loss and PSNR evolve during training?
4. **When would you expect ZSSR to fail** — e.g. for which image types, scale factors, or noise levels — and why?

*Your analysis here:*

# Part 9: Bonus — Advanced Extensions

For **bonus credit**, choose **at least one** of the following techniques (from the ZSSR paper and beyond), implement it in the cell below, run it, and **briefly discuss its effect** on PSNR compared with your basic ZSSR model from Part 8. Pick whichever interests you most — you don't need to do more than one. **Read the paper** (Shocher et al., CVPR 2018); most of these are described there.

- **Back-projection.** After getting an SR estimate, enforce LR-consistency: downscale the SR back to the LR size, compute the error against the original LR, upscale that error, and add it back. Repeat for a few iterations.

- **Gradual / progressive SR.** Instead of reaching the target scale in one step, upscale in stages (e.g. ×2 then ×2 for ×4), and/or add training pairs at intermediate scales so the network sees more diverse (son, father) examples.

- **Architecture exploration.** Try deeper / wider networks (vary `n_channels` / `n_layers`), skip connections, or different activations, and report how capacity affects the PSNR improvement. Does more capacity always help on a single image?

- **Loss-function experiments.** L1 is the standard baseline (Part 4) — try L2, a combination, or a gradient-domain loss, and compare the resulting PSNR.

*Keep it focused: a clear implementation plus a short comparison (numbers and/or a plot) and a sentence or two of discussion is enough for full bonus credit.*

In [ ]:
# --- Bonus: your implementation here ---
# Choose at least one idea above, implement it, run it, and print/plot a comparison
# against your basic ZSSR result from Part 8.


### Bonus discussion

*Which idea(s) did you implement, and what did you find? (a few sentences)*

# Conclusion

You have successfully implemented Zero-Shot Super Resolution (ZSSR), a self-supervised learning approach that:

1. **Learns from a single image**: No large training datasets required
2. **Exploits internal patch recurrence**: The key insight that makes ZSSR work
3. **Trains at test time**: Adapts to the specific image being super-resolved
4. **Achieves competitive results**: Often outperforms generic bicubic upsampling

## Key Takeaways

- **Self-supervised learning**: Can train without paired data by exploiting image self-similarity
- **Image-specific adaptation**: The best model for one image may not be best for another
- **Practical deep learning**: Shows that deep learning can work with minimal data in the right setting

## Further Reading

- Original ZSSR paper: Shocher et al., "Zero-Shot Super-Resolution using Deep Internal Learning" (CVPR 2018)
- ResizeRight library: https://github.com/assafshocher/ResizeRight
- Related work: Internal Generative Model Exploitation (IGeMM), Hallucinating Images (HI)

## Before you submit

- Fill in your `STUDENT_ID` at the top of the notebook.
- Run all cells top-to-bottom (Kernel → Restart & Run All) so everything executes in order.
- **Save the notebook with its outputs** (figures and printouts visible).
- Submit the `.ipynb` file.

Good luck!